# Try to use pytorch on the PPMI data

First import libs and define paths

In [1]:
import pandas as pd
import os

In [2]:

base_path = "/home/data/PPMI"
participants_file = os.path.join(base_path, "rawdata/participants.tsv")
curated_file = os.path.join(base_path, "documents/PPMI_Curated_Data_Cut_Public_20240729.xlsx")

In [3]:
# ----------------------- mirar tsv
print("Info on participants.tsv")
try:
    df_p = pd.read_csv(participants_file, sep='\t')
    print("Total subjects: ", len(df_p))

    # Look for a 'diagnosis' or 'group' column
    if 'diagnosis' in df_p.columns:
        print(df_p['diagnosis'].value_counts())
    else:
        print("Columns available:", df_p.columns.tolist())
except Exception as e:
    print("Could not read participants.tsv: ", e)

Info on participants.tsv
Total subjects:  955
Columns available: ['participant_id', 'cohort', 'sex', 'birth_date', 'education', 'handed']


In [4]:
# ----------------------- mirar excel
print("Info on the excel")
try:
    df_c = pd.read_excel(curated_file)
    print(f"Total entries in curated data: {len(df_c)}")

    # Common PPMI diagnosis labels: PD (Parkinson's), HC (Healthy Control), SWEDD
    if 'COHORT_DEFINITION' in df_c.columns:
        print(df_c['COHORT_DEFINITION'].value_counts())
    else:
        # If the column name is different lemme see them
        print("Columns available:", df_c.columns.tolist()[:10], "...(truncated)")

except Exception as e:
    print(f"Could not read Excel file: {e}")

Info on the excel
Total entries in curated data: 13242
Columns available: ['SITE', 'PATNO', 'COHORT', 'subgroup', 'enroll_phase', 'HIQ_RBD', 'study_status', 'NSD_Status', 'NSD_STAGE', 'PRIMDIAG'] ...(truncated)


# Pytorch

If all above works, load `pytorch` and `nibabel`.

In [5]:
import torch
from torch.utils.data import Dataset
import nibabel as nib
import numpy as np

pytorch needs a dataset class to handle the "fetching" of stuff like which patient is and is not sick.

In [6]:
class PPMIDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # Load the 3D NIfTI file
        img = nib.load(self.file_paths[idx]).get_fdata()

        # Preprocessing
        img = (img - np.mean(img)) / np.std(img) # Normalize intensity

        # Convert to Torch Tensor (Add a 'channel' dimension for the CNN)
        # Shape becomes: [1, Depth, Height, Width]
        img_tensor = torch.from_numpy(img).float().unsqueeze(0)

        label = torch.tensor(self.labels[idx])
        return img_tensor, label

In [8]:
import glob

In [24]:
derivatives_path = os.path.join(base_path, "derivatives/dat-reg-v6")
excel_path = os.path.join(base_path, "documents/PPMI_Curated_Data_Cut_Public_20240729.xlsx")

# Load the labels
df = pd.read_excel(excel_path)

# agafam només el num del pacient i el seu cohort
labels_map = df[['PATNO', 'COHORT']].drop_duplicates().set_index('PATNO')['COHORT'].to_dict()
#print("labels map: ", labels_map)

# trobam totes les "ses-BL" de DaTscan
baseline_images = glob.glob(f"{derivatives_path}/sub-*/ses-BL/spect/*_DaTSCAN.nii.gz") # màgia negra
#print("baseline_images: ", baseline_images)

data_list = []

for img_path in baseline_images:

    # agafam PATNO com a int del filename (aka sub-PPMI100001 -> 100001)
    sub_id = img_path.split('/')[-4] # només 'sub-PPMI100001'
    patno = int(sub_id.replace('sub-PPMI', ''))

    # haurien d ser iguals
    #print("img_path:\t", img_path)
    #print("patno:\t", patno)

    # agafam el cohort si el pacient tenia metadades al excel
    # segur q pandas té algo més ràpid x fer això
    if patno in labels_map:
        cohort = labels_map[patno]
        # x ara només interessen els PD i healthy
        if cohort in [1, 2]: #  cohort 1 són els PD i cohort 2 són els healthy
            label = 1 if cohort == 1 else 0 # els PD es marquen com 1 i els healthy es posen a 0
            #data_list.append({'path': img_path, 'label': label, 'cohort': cohort})
            data_list.append({'path': img_path, 'label': label})

# Create a final clean CSV for training
clean_df = pd.DataFrame(data_list)
print("clean_df:")
print(clean_df.head(10))
print(clean_df['label'].value_counts())
clean_df.to_csv("ppmi_baseline_mapping.csv", index=False)
print("\nMapping saved to 'ppmi_baseline_mapping.csv'!")

clean_df:
            label
count  685.000000
mean     0.818978
std      0.385318
min      0.000000
25%      1.000000
50%      1.000000
75%      1.000000
max      1.000000
                                                path  label
0  /home/data/PPMI/derivatives/dat-reg-v6/sub-PPM...      1
1  /home/data/PPMI/derivatives/dat-reg-v6/sub-PPM...      1
2  /home/data/PPMI/derivatives/dat-reg-v6/sub-PPM...      1
3  /home/data/PPMI/derivatives/dat-reg-v6/sub-PPM...      0
4  /home/data/PPMI/derivatives/dat-reg-v6/sub-PPM...      1
5  /home/data/PPMI/derivatives/dat-reg-v6/sub-PPM...      1
6  /home/data/PPMI/derivatives/dat-reg-v6/sub-PPM...      1
7  /home/data/PPMI/derivatives/dat-reg-v6/sub-PPM...      1
8  /home/data/PPMI/derivatives/dat-reg-v6/sub-PPM...      1
9  /home/data/PPMI/derivatives/dat-reg-v6/sub-PPM...      0
label
1    561
0    124
Name: count, dtype: int64

Mapping saved to 'ppmi_baseline_mapping.csv'!


Now we have a cleaned-up dataset with only PD or not.

# Train-test split

Now scikit learn comes in to train-split the dataset

In [26]:
from sklearn.model_selection import train_test_split

df = pd.read_csv("ppmi_baseline_mapping.csv")
# 80% Train, 20% Test/Validation for now?
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=69)

print(f"Original df: {len(df)} samples")
print(f"Training on: {len(train_df)} samples")
print(f"Testing on: {len(test_df)} samples")

Original df: 685 samples
Training on: 548 samples
Testing on: 137 samples


# CNN

And once that's done, onto creating the CNN


In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ParkinsonClassifier3D(nn.Module):
    def __init__(self):
        super(ParkinsonClassifier3D, self).__init__()
        # Input: 1 channel (scan), Output: 16 filters
        self.conv1 = nn.Conv3d(1, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool3d(2)

        self.conv2 = nn.Conv3d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv3d(32, 64, kernel_size=3, padding=1)

        # This part depends on image dimensions (91x109x91, isnt it?)
        # Global Average Pooling to avoid calculating flat dimensions
        self.gap = nn.AdaptiveAvgPool3d(1)

        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 1) # Binary output (0 or 1)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = self.gap(x) # Flattens to [Batch, 64, 1, 1, 1]
        x = x.view(-1, 64) # Flattens to [Batch, 64]

        x = F.relu(self.fc1(x))
        # sigmoid for binary classification probability bcs why not
        x = torch.sigmoid(self.fc2(x))
        return x

model = ParkinsonClassifier3D().to("cuda") # Send to your GPU!#%%


In [40]:
######################### testing nibabel

print("1st training image")

ist_path = train_df.iloc[0]['path']
ist_label = train_df.iloc[0]['label']

print("path : ", ist_path)
print("label: ", ist_label)

ist_img = nib.load(ist_path)
ist_img_data = ist_img.get_fdata()
print("Shape: ", ist_image_data.shape)

print("ist_img_data is float64: ", ist_img_data.dtype == np.dtype(np.float64))

# Normalize
ist_img_data = (ist_img_data - ist_img_data.mean()) / (ist_img_data.std() + 1e-8)

# Convert to tensor and add channel dim: [1, D, H, W]
img_tensor = torch.from_numpy(ist_img_data).float().unsqueeze(0)
label_tensor = torch.tensor(ist_label).float().unsqueeze(0)

print("img_tensor shape: ", img_tensor.shape)
print("label: ", label_tensor)


1st training image
path :  /home/data/PPMI/derivatives/dat-reg-v6/sub-PPMI3320/ses-BL/spect/sub-PPMI3320_ses-BL_desc-resampled_DaTSCAN.nii.gz
label:  0
Shape:  (176, 240, 256)
ist_img_data is float64:  True
img_tensor shape:  torch.Size([1, 176, 240, 256])
label:  tensor([0.])


In [39]:
import nibabel as nib
from torch.utils.data import Dataset, DataLoader

class DaTscanDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.df.iloc[idx]['path']
        label = self.df.iloc[idx]['label']

        # Load NIfTI and get data
        img = nib.load(path) # now img is an instance of a nibabel image. (docs)

        img = img.get_fdata()

        # Normalize
        img = (img - img.mean()) / (img.std() + 1e-8)

        # Convert to tensor and add channel dim: [1, D, H, W]
        img_tensor = torch.from_numpy(img).float().unsqueeze(0)
        label_tensor = torch.tensor(label).float().unsqueeze(0)

        return img_tensor, label_tensor

# Create the loaders
train_loader = DataLoader(DaTscanDataset(train_df), batch_size=4, shuffle=True)
test_loader = DataLoader(DaTscanDataset(test_df), batch_size=4)

Ok finaly: now we trainin

In [41]:
import torch.optim as optim

In [42]:
# Initialize Model, Loss, and Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ParkinsonClassifier3D().to(device)

# Use BCEWithLogitsLoss because it is more numerically stable than putting a Sigmoid inside the model.
# NOTE: If you use this, REMOVE the 'torch.sigmoid' from your model's forward function!
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# We Trainin
num_epochs = 10

print(f"Starting training on {device}...")

for epoch in range(num_epochs):
    model.train() # Set model to training mode
    running_loss = 0.0

    for i, (images, labels) in enumerate(train_loader):
        # Move data to GPU
        images = images.to(device)
        labels = labels.to(device)

        # Zero the gradients (clear the previous round's memory)
        optimizer.zero_grad()

        # Forward pass: Get predictions
        outputs = model(images)

        # Calculate Loss
        loss = criterion(outputs, labels)

        # Backward pass: Calculate how to improve
        loss.backward()

        # Step: Actually update the weights
        optimizer.step()

        running_loss += loss.item()

        if (i + 1) % 5 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

    print(f"Epoch {epoch+1} average loss: {running_loss/len(train_loader):.4f}")

print("Training Complete!")

Starting training on cuda...


RuntimeError: stack expects each tensor to be equal size, but got [1, 176, 256, 256] at entry 0 and [1, 150, 240, 240] at entry 1